# Customer Churn Prediction 📉

**Goal:** Predict which telecom customers are likely to leave using Logistic Regression, Random Forest, and XGBoost.

This notebook covers data loading, EDA, preprocessing, model training, evaluation (accuracy, precision, recall, ROC-AUC), and basic feature importance.

**Dataset:** IBM Telco Customer Churn. Download the CSV from [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn), unzip it, and upload `WA_Fn-UseC_-Telco-Customer-Churn.csv` when prompted below. Do not upload customer data to GitHub.

> This is an educational baseline. Model scores are not guaranteed and the best threshold depends on business costs.

## 1. Install and import libraries

In [ ]:
!pip -q install xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

## 2. Upload and load the dataset

Run this cell and choose the CSV file downloaded from Kaggle. If your filename is different, the notebook will use the uploaded filename automatically.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError('Please upload the Telco churn CSV file.')
csv_filename = next(iter(uploaded))
df = pd.read_csv(csv_filename)
print('Loaded:', csv_filename)
print('Shape:', df.shape)
display(df.head())

## 3. Inspect and clean data

In [ ]:
df.info()
display(df.isna().sum().sort_values(ascending=False).head(15))
print('Duplicate rows:', df.duplicated().sum())
print('Columns:', df.columns.tolist())

# Convert TotalCharges to numeric; blank strings become missing values.
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

if 'Churn' not in df.columns:
    raise ValueError("Target column 'Churn' was not found. Check that you uploaded the correct dataset.")
print('Target values:', df['Churn'].value_counts(dropna=False))

## 4. Exploratory data analysis (EDA)

In [ ]:
# Target distribution
print((df['Churn'].value_counts(normalize=True) * 100).round(2).rename('percent'))
sns.countplot(data=df, x='Churn')
plt.title('Customer Churn Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Churn rates by a few important categorical features
for col in ['Contract', 'InternetService', 'PaymentMethod', 'PaperlessBilling']:
    if col in df.columns:
        rate = df.groupby(col, dropna=False)['Churn'].apply(
            lambda s: (s.astype(str).str.strip() == 'Yes').mean() * 100
        ).sort_values(ascending=False)
        print(f'\nChurn rate by {col} (%)')
        display(rate.round(2).to_frame('Churn rate (%)'))
        rate.plot(kind='bar', title=f'Churn rate by {col}', ylabel='Churn rate (%)')
        plt.xticks(rotation=35, ha='right')
        plt.tight_layout()
        plt.show()

In [ ]:
# Numeric distributions (if these columns exist)
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    if col in df.columns:
        sns.boxplot(data=df, x='Churn', y=col)
        plt.title(f'{col} by Churn')
        plt.tight_layout()
        plt.show()

## 5. Prepare features and target

We remove the customer identifier, map the target to 0/1, split before fitting imputers/encoders, and keep preprocessing inside a pipeline to reduce data leakage.

In [ ]:
# Remove identifier if present (it usually has no generalizable predictive value).
df_model = df.drop(columns=['customerID'], errors='ignore').copy()
df_model = df_model.dropna(subset=['Churn'])
df_model['Churn'] = df_model['Churn'].map({'No': 0, 'Yes': 1})
df_model = df_model.dropna(subset=['Churn'])

X = df_model.drop(columns=['Churn'])
y = df_model['Churn'].astype(int)

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)

## 6. Build preprocessing and models

In [ ]:
def make_preprocessor():
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ])

# Account for class imbalance for the models that support it.
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / max(positive_count, 1)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1
    )
}

trained_models = {}
for name, model in models.items():
    pipe = Pipeline([('preprocessor', make_preprocessor()), ('classifier', model)])
    pipe.fit(X_train, y_train)
    trained_models[name] = pipe
    print('Trained:', name)

## 7. Compare model performance

Metrics below are calculated on the held-out test split. Because these results are used for comparison in this educational notebook, use cross-validation on the training set for more rigorous tuning and reserve the test set for the final evaluation in a production workflow.

In [ ]:
results = []
for name, pipe in trained_models.items():
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
display(results_df.round(3))

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, pipe in trained_models.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle='--', label='Chance')
ax.set_title('ROC Curve Comparison')
plt.tight_layout()
plt.show()

## 8. Inspect one model's classification results

For illustration, this selects the highest ROC-AUC model from the comparison. For formal reporting, select/tune models using training-set cross-validation and evaluate the final chosen model once on the test set.

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]
print('Selected for illustration:', best_name)
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['No Churn', 'Churn'], cmap='Blues'
)
plt.title(f'Confusion Matrix - {best_name}')
plt.tight_layout()
plt.show()

## 9. Explore a lower probability threshold

The default threshold is usually 0.5. Lower thresholds often increase recall but can also create more false positives. Choose a business threshold on validation data, not by repeatedly optimizing on the test set.

In [ ]:
for threshold in [0.30, 0.40, 0.50, 0.60]:
    pred_threshold = (y_prob >= threshold).astype(int)
    print(
        f'Threshold={threshold:.2f} | '
        f'Accuracy={accuracy_score(y_test, pred_threshold):.3f} | '
        f'Precision={precision_score(y_test, pred_threshold, zero_division=0):.3f} | '
        f'Recall={recall_score(y_test, pred_threshold, zero_division=0):.3f}'
    )

## 10. Feature importance (tree-based model)

Feature importance shows which transformed features the Random Forest uses most. It does not prove causation or necessarily explain an individual prediction.

In [ ]:
rf_pipe = trained_models['Random Forest']
feature_names = rf_pipe.named_steps['preprocessor'].get_feature_names_out()
importances = rf_pipe.named_steps['classifier'].feature_importances_
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=False).head(15)
display(importance_df)
importance_df.sort_values('Importance').plot(
    x='Feature', y='Importance', kind='barh', legend=False, figsize=(9, 6)
)
plt.title('Top 15 Random Forest Feature Importances')
plt.tight_layout()
plt.show()

## 11. Create a customer risk output

Risk scores are estimates, not certainties. Use them to prioritize supportive outreach and evaluate whether retention actions work.

In [ ]:
customer_risk = X_test.copy()
customer_risk['Actual_Churn'] = y_test
customer_risk['Churn_Probability'] = y_prob
customer_risk['Predicted_Churn_At_0.5'] = (y_prob >= 0.5).astype(int)
customer_risk = customer_risk.sort_values('Churn_Probability', ascending=False)
display(customer_risk.head(20))

# Optional: download result file to your computer; don't commit customer-level data to GitHub.
customer_risk.to_csv('customer_churn_predictions.csv', index=False)
print('Saved customer_churn_predictions.csv in the Colab session.')

## 12. Conclusions (complete after running)

In your report, write down: (1) observed churn rate, (2) the model with the best validation/test metrics, (3) recall and ROC-AUC, (4) strongest model features, (5) proposed retention actions, and (6) limitations. Do not copy example scores from tutorials; report your own run's results.

**Limitations:** This is a historical dataset; relationships may change. The model is predictive rather than causal, and a real deployment requires privacy review, drift monitoring, calibration, threshold/cost analysis, and evaluation of retention experiments.